# M1 Notebook 17 — Estimation and Confidence Intervals

**Notebook ID:** M1_N17  
**Status:** Runnable first edition  
**Random seed:** 42

> Point estimates summarize what the sample suggests. Confidence intervals quantify the uncertainty around those estimates under stated assumptions.


## 1. Learning objectives

1. Distinguish parameters from estimators and estimates.
2. Compute sample means, variances, and standard errors.
3. Construct normal and Student-t confidence intervals for means.
4. Construct Wilson intervals for proportions.
5. Construct percentile bootstrap intervals.
6. Interpret confidence level and repeated-sampling coverage.
7. Determine sample sizes for desired margins of error.


In [ ]:
from srai_math.utils import environment_info, set_seed
from srai_math.statistics import (
    bootstrap_confidence_interval,
    normal_mean_confidence_interval,
    proportion_confidence_interval_wilson,
    required_sample_size_mean,
    required_sample_size_proportion,
    sample_mean,
    sample_variance,
    standard_error_mean,
    t_mean_confidence_interval,
)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
set_seed(42)
environment_info()


## 2. Parameters, estimators, and estimates

A parameter is a fixed but usually unknown population quantity, such as \(\mu\).

An estimator is a rule, such as

\[
\bar X=\frac{1}{n}\sum_{i=1}^n X_i.
\]

An estimate is the numerical value produced by the estimator for one sample.


## 3. Point estimation

In [ ]:
sample = np.array([12.0, 15.0, 14.0, 18.0, 22.0, 19.0, 16.0, 17.0])

summary = {
    "sample_mean": sample_mean(sample),
    "sample_variance": sample_variance(sample),
    "standard_error_mean": standard_error_mean(sample),
}
summary


## 4. Confidence interval with known population standard deviation

When \(\sigma\) is known,

\[
\bar x
\pm
z_{1-\alpha/2}\frac{\sigma}{\sqrt n}.
\]


In [ ]:
known_sigma_interval = normal_mean_confidence_interval(
    mean=50.0,
    population_std=10.0,
    sample_size=100,
    confidence=0.95,
)
known_sigma_interval


## 5. Student-t confidence interval

When \(\sigma\) is unknown,

\[
\bar x
\pm
t_{1-\alpha/2,n-1}
\frac{s}{\sqrt n}.
\]


In [ ]:
t_interval = t_mean_confidence_interval(sample, confidence=0.95)

{
    "estimate": sample.mean(),
    "95_percent_t_interval": t_interval,
}


## 6. Confidence interval interpretation

A 95% confidence procedure has approximately 95% repeated-sampling coverage under its assumptions.

It does not mean that, after the interval is computed, there is a 95% frequentist probability that the fixed parameter lies inside that particular interval.


## 7. Repeated-sampling coverage experiment

In [ ]:
rng = np.random.default_rng(42)
true_mean = 10.0
true_std = 3.0
sample_size = 30
repetitions = 3000

covered = 0
intervals = []

for i in range(repetitions):
    x = rng.normal(true_mean, true_std, sample_size)
    lower, upper = t_mean_confidence_interval(x, confidence=0.95)
    covered += lower <= true_mean <= upper
    if i < 100:
        intervals.append((lower, upper, x.mean()))

coverage = covered / repetitions
coverage


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for i, (lower, upper, estimate) in enumerate(intervals):
    ax.plot([lower, upper], [i, i])
    ax.scatter([estimate], [i], s=10)
ax.axvline(true_mean, linestyle="--")
ax.set_xlabel("Mean estimate and interval")
ax.set_ylabel("Simulation index")
ax.set_title("First 100 Repeated-Sampling Confidence Intervals")
plt.show()


## 8. Interval width and sample size

In [ ]:
sample_sizes = np.array([10, 25, 50, 100, 250, 500])
widths = []

for n in sample_sizes:
    lower, upper = normal_mean_confidence_interval(
        mean=0.0,
        population_std=10.0,
        sample_size=int(n),
        confidence=0.95,
    )
    widths.append(upper - lower)

pd.DataFrame({
    "sample_size": sample_sizes,
    "interval_width": widths,
})


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(sample_sizes, widths, marker="o")
ax.set_xlabel("Sample size")
ax.set_ylabel("95% interval width")
ax.set_title("Confidence Intervals Narrow with Larger Samples")
plt.show()


## 9. Confidence interval for a proportion

The Wilson interval generally behaves better than the simple Wald interval, especially with smaller samples or proportions near zero or one.


In [ ]:
successes = 42
trials = 80

wilson_interval = proportion_confidence_interval_wilson(
    successes,
    trials,
    confidence=0.95,
)

{
    "sample_proportion": successes/trials,
    "Wilson_95_percent_interval": wilson_interval,
}


## 10. Bootstrap confidence interval

In [ ]:
skewed_sample = np.array([5, 6, 7, 8, 9, 10, 12, 15, 25, 40], dtype=float)

bootstrap_interval = bootstrap_confidence_interval(
    skewed_sample,
    statistic=np.median,
    confidence=0.95,
    repetitions=10_000,
    seed=42,
)

{
    "sample_median": np.median(skewed_sample),
    "bootstrap_95_percent_interval": bootstrap_interval,
}


## 11. Visualizing a bootstrap distribution

In [ ]:
rng = np.random.default_rng(42)
bootstrap_medians = np.empty(5000)

for i in range(5000):
    bootstrap_sample = rng.choice(
        skewed_sample,
        size=skewed_sample.size,
        replace=True,
    )
    bootstrap_medians[i] = np.median(bootstrap_sample)

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(bootstrap_medians, bins=30, density=True)
ax.axvline(np.median(skewed_sample), linestyle="--")
ax.set_xlabel("Bootstrap median")
ax.set_ylabel("Density")
ax.set_title("Bootstrap Distribution of the Median")
plt.show()


## 12. Required sample size for a mean

In [ ]:
mean_sample_size = required_sample_size_mean(
    population_std=15.0,
    margin_error=2.0,
    confidence=0.95,
)
mean_sample_size


## 13. Required sample size for a proportion

In [ ]:
proportion_sample_sizes = pd.DataFrame({
    "anticipated_proportion": [0.1, 0.3, 0.5],
    "required_sample_size": [
        required_sample_size_proportion(
            margin_error=0.05,
            confidence=0.95,
            anticipated_proportion=p,
        )
        for p in [0.1, 0.3, 0.5]
    ],
})
proportion_sample_sizes


Using \(p=0.5\) gives the largest variance and is therefore conservative when no prior proportion estimate is available.


## 14. Statistics interpretation

Estimation and confidence intervals form the bridge from sample data to uncertain population statements. Their validity depends on design, assumptions, and estimator properties.


## 15. AI interpretation

Confidence intervals and related uncertainty tools support:

- model-performance comparison;
- prediction uncertainty;
- bootstrap ensembles;
- calibration assessment;
- A/B testing;
- monitoring drift and degradation.


## 16. Decision Intelligence case — Estimating service coverage

Suppose a survey finds that 420 of 600 sampled households have access to a service.


In [ ]:
covered_households = 420
surveyed_households = 600

coverage_estimate = covered_households / surveyed_households
coverage_interval = proportion_confidence_interval_wilson(
    covered_households,
    surveyed_households,
    confidence=0.95,
)

{
    "coverage_estimate": coverage_estimate,
    "95_percent_interval": coverage_interval,
}


### Interpretation

The interval quantifies sampling uncertainty under a binomial-style model. A real survey may also require weights, stratification, clustering, nonresponse adjustment, and design-based variance estimation.


## 17. Engineering notes

- Confidence intervals quantify sampling uncertainty, not all uncertainty.
- Complex surveys need design-consistent standard errors.
- Bootstrap resampling must respect the data structure.
- Narrow intervals can still center on biased estimates.
- Multiple intervals require multiplicity awareness.
- Missing data and measurement error may dominate sampling error.


## 18. Common errors

- Treating confidence level as posterior probability.
- Confusing standard deviation with standard error.
- Using a normal interval with tiny samples and strong skewness.
- Ignoring finite population and survey-design effects.
- Choosing a sample size without accounting for nonresponse.
- Reporting estimates without intervals.


## 19. Exercises

### Level A
Distinguish parameter, estimator, and estimate.

### Level B
Derive the normal confidence interval for a mean.

### Level C
Simulate repeated-sampling coverage for normal and skewed populations.

### Capstone
Estimate a public-service indicator, select an appropriate interval, justify the sample size, and document all sources of uncertainty not captured by the interval.


## 20. Key insight

Point estimates summarize the sample. Confidence intervals describe the repeated-sampling uncertainty of an estimation procedure. Good decision support reports both the estimate and the uncertainty around it.
